In [27]:
import polars as pl
import pandas as pd
import time

In [9]:
lf = pl.scan_csv("steam_top_games_2026.csv");

In [26]:
df = pd.read_csv("steam_top_games_2026.csv")

In [21]:
lf.collect_schema().names()

['app_id',
 'name',
 'release_date',
 'coming_soon',
 'price_usd',
 'is_free',
 'discount_pct',
 'developer',
 'publisher',
 'genres',
 'categories',
 'tags',
 'platforms_win',
 'platforms_mac',
 'platforms_linux',
 'metacritic_score',
 'recommendations',
 'positive_reviews',
 'negative_reviews',
 'estimated_owners',
 'avg_playtime_forever',
 'avg_playtime_2weeks',
 'median_playtime',
 'peak_ccu',
 'required_age',
 'dlc_count',
 'achievements',
 'short_description',
 'header_image']

In [55]:
paid_stats = (
    lf.filter(pl.col('is_free') == False)
    .select(pl.col("price_usd"))
    .collect()
    .describe()
)
paid_stats

statistic,price_usd
str,f64
"""count""",1246.0
"""null_count""",0.0
"""mean""",14.454029
"""std""",14.054226
"""min""",0.0
"""25%""",3.99
"""50%""",9.99
"""75%""",19.99
"""max""",149.99


In [23]:
results = (
    lf.group_by("genres")
    .agg(
        pl.len().alias("total_games"),
        pl.col("price_usd").mean().alias("avg_price"),
        pl.col("peak_ccu").max().alias("highest_peak"),
        pl.col("peak_ccu").min().alias("lowest_peak")
    )
    .sort("total_games", descending=True)
    .collect()
)

results

genres,total_games,avg_price,highest_peak,lowest_peak
str,u32,f64,i64,i64
"""Action""",108,17.497222,44886,0
"""Action, Adventure""",72,21.97,14920,0
"""Action, Indie""",71,12.492817,2080,0
"""Adventure, Indie""",58,10.518966,498,0
"""Action, Adventure, Indie""",54,14.837407,3743,0
…,…,…,…,…
"""Action, Casual, Indie, Strateg…",1,0.0,0,0
"""Free To Play, Massively Multip…",1,0.0,0,0
"""Adventure, Casual, RPG, Simula…",1,0.0,3279,3279


In [41]:
start_time = time.time()
comparison = (
    lf.group_by("is_free")
    
    .agg(
        pl.col("peak_ccu").mean().alias("avg_peak_ccu"),
        pl.len().alias("game_count")
    )
    
    .collect()
)
end_time = time.time()

end_time - start_time

0.0033295154571533203

In [45]:
start_time = time.time()

avg_ccu = df.groupby("is_free")["peak_ccu"].agg(
    avg_peak_ccu='mean',
    game_count='size'
)

end_time = time.time()

end_time - start_time

0.0010263919830322266